# Conhecendo o pandas
Primeiros passos com a biblioteca pandas usando uma base de imóveis para aluguel no Rio de Janeiro: leitura, inspeção, agrupamentos, gráficos, tratamento de nulos, filtros, criação de colunas e exportação. No final, os desafios do curso com uma base de alunos.

**Origem:** Alura — curso "Pandas: conhecendo a biblioteca" (formação Python para Data Science).

**Dados (lidos direto da URL):**
- Aluguéis: `https://raw.githubusercontent.com/alura-cursos/pandas-conhecendo-a-biblioteca/main/base-de-dados/aluguel.csv`
- Desafios: `https://raw.githubusercontent.com/alura-cursos/pandas-conhecendo-a-biblioteca/main/desafios/alunos.csv`

**Índice**
1. Leitura e inspeção dos dados
2. Análise exploratória por tipo de imóvel
3. Tratamento de valores nulos e remoção de registros
4. Filtros com seleções booleanas
5. Salvando os dados
6. Criando novas colunas
7. Desafios com a base de alunos

In [ ]:
# Setup (pandas e matplotlib, usado pelos gráficos .plot, já vêm no Colab)
import pandas as pd

## 1. Leitura e inspeção dos dados
`read_csv` lê o arquivo (separado por `;`). Guardamos a base original em `dados` e trabalhamos numa cópia `df`, para poder recomeçar a análise mais adiante sem baixar o arquivo de novo.

In [ ]:
url = 'https://raw.githubusercontent.com/alura-cursos/pandas-conhecendo-a-biblioteca/main/base-de-dados/aluguel.csv'
dados = pd.read_csv(url, sep=';')
df = dados.copy()
df.head(10)

In [ ]:
df.tail(10)

In [ ]:
print(type(df))     # tipo do dado: DataFrame
print(df.shape)     # formato do DataFrame: (linhas, colunas)
print(df.columns)   # nomes das colunas
df.info()           # tipos e quantidade de valores não nulos por coluna

In [ ]:
# Uma coluna é uma Series (uma coluna com índice)
df['Tipo']

In [ ]:
# Para selecionar várias colunas usa-se colchetes duplos: a lista de colunas vai dentro do []
df[['Quartos', 'Valor']]

In [ ]:
# Estatísticas das colunas numéricas
df.describe()

## 2. Análise exploratória por tipo de imóvel
`groupby` agrupa as linhas por uma coluna e aplica uma agregação; com colchetes duplos (`[['Valor']]`) o resultado continua sendo um DataFrame, o que facilita ordenar e plotar.

In [ ]:
# Média do valor do aluguel por tipo, em ordem crescente
df_preco_tipo = df.groupby('Tipo')[['Valor']].mean().sort_values('Valor')
df_preco_tipo.plot(kind='barh', figsize=(16, 10), color='purple')

In [ ]:
# Removendo os imóveis comerciais (o @ referencia uma variável dentro do query)
df.Tipo.unique()

In [ ]:
imoveis_comerciais = ['Conjunto Comercial/Sala',
                      'Prédio Inteiro',
                      'Loja/Salão',
                      'Galpão/Depósito/Armazém',
                      'Casa Comercial',
                      'Terreno Padrão',
                      'Loja Shopping/ Ct Comercial',
                      'Box/Garagem',
                      'Chácara',
                      'Loteamento/Condomínio',
                      'Sítio',
                      'Pousada/Chalé',
                      'Hotel',
                      'Indústria']

df = df.query('@imoveis_comerciais not in Tipo')
df.Tipo.unique()

In [ ]:
# Média por tipo, agora só com imóveis residenciais
df.groupby('Tipo')[['Valor']].mean().sort_values('Valor').plot(kind='barh', figsize=(16, 10), color='purple')

In [ ]:
# Percentual de cada tipo de imóvel
df_percentual_tipo = df.Tipo.value_counts(normalize=True).to_frame().sort_values('Tipo')
df_percentual_tipo = df_percentual_tipo.rename(columns={'proportion': 'Percentuais'})
df_percentual_tipo.plot(kind='bar', figsize=(14, 10), color='green', edgecolor='white',
                        xlabel='Tipo', ylabel='Percentual')

In [ ]:
# Como a maioria é apartamento, seguimos só com esse tipo
df = df.query('Tipo == "Apartamento"')
df.Tipo.unique()

In [ ]:
print(df.Quartos.mean())            # média de quartos
print(len(df['Bairro'].unique()))   # quantidade de bairros
media_geral = df.Valor.mean()       # valor médio do aluguel
media_geral

In [ ]:
# Os 5 bairros com maior aluguel médio, mantendo apenas os que ficam acima da média geral
media_bairros = df.groupby('Bairro')[['Valor']].mean().sort_values('Valor', ascending=False).head()
bairros_maiores_que_media = media_bairros.query('Valor > @media_geral')
bairros_maiores_que_media.plot(kind='barh', figsize=(14, 10), color='red', xlabel='Valor', ylabel='Bairro')

## 3. Tratamento de valores nulos e remoção de registros
Estratégias para nulos: preencher com uma constante (`fillna(0)`), com o valor anterior (`ffill`), com o próximo (`bfill`) ou interpolando os vizinhos (`interpolate`). Aqui usamos `fillna(0)`, o que permite identificar e remover depois os apartamentos sem valor ou sem condomínio.

In [ ]:
df.isnull().sum()   # quantidade de nulos por coluna

In [ ]:
df = df.fillna(0)   # preenche os nulos com 0

# Alternativas (não aplicadas aqui):
# df = df.ffill()                                              # valor da linha anterior
# df = df.bfill()                                              # valor da linha seguinte
# df['Condominio'] = df['Condominio'].interpolate(method='linear')  # média dos vizinhos

df.isnull().sum()

In [ ]:
# Remove apartamentos sem valor de aluguel ou sem condomínio
registros_a_remover = df.query('Valor == 0 | Condominio == 0').index
df = df.drop(registros_a_remover, axis=0)

# A coluna Tipo agora tem uma única categoria (Apartamento), então pode ser removida
df = df.drop('Tipo', axis=1)
df

## 4. Filtros com seleções booleanas
Cada condição gera uma Series de `True`/`False`; combinamos com `&` (e) e `|` (ou), sempre com parênteses.

In [ ]:
# Apartamentos com 1 quarto e aluguel menor que R$ 1200
selecao1 = df['Quartos'] == 1
selecao2 = df['Valor'] < 1200
selecao_final = (selecao1) & (selecao2)
df_selecionada = df[selecao_final]
df_selecionada

In [ ]:
# Pelo menos 2 quartos, aluguel menor que R$ 1200 e área maior que 70 m²
selecao_2 = (df['Quartos'] >= 2) & (df['Valor'] < 1200) & (df['Area'] > 70)
df[selecao_2].head()

In [ ]:
# Área maior que 80 m² ou aluguel menor que R$ 4000
selecao_3 = (df['Area'] > 80) | (df['Valor'] < 4000)
df[selecao_3]

## 5. Salvando os dados

In [ ]:
df.to_csv('dados_apartamentos.csv', index=False, sep=';')

## 6. Criando novas colunas
Recomeçamos a partir da base original completa (`dados`) e criamos colunas a partir de operações entre colunas, concatenação de textos e `apply` com `lambda`.

In [ ]:
df = dados.copy()

df['Valor_por_mes'] = df['Valor'] + df['Condominio']           # aluguel + condomínio
df['Valor_por_ano'] = df['Valor_por_mes'] * 12 + df['IPTU']
df.head()

In [ ]:
df['Descricao'] = (df['Tipo'] + ' em ' + df['Bairro'] + ' com ' + df['Quartos'].astype(str) +
                   ' quarto(s) e ' + df['Vagas'].astype(str) + ' vaga(s) de garagem.')
df.Descricao.head()

In [ ]:
df['Possui_suite'] = df['Suites'].apply(lambda x: 'Sim' if x > 0 else 'Não')
df.Possui_suite.head()

In [ ]:
df.to_csv('dados_completos_dev.csv', index=False, sep=';')

### 6.1 Formas de criar uma coluna
Quatro maneiras de criar a coluna `C` num DataFrame pequeno: atribuindo uma lista, somando colunas, com `assign` e com `apply`.

In [ ]:
df0 = pd.DataFrame({'A': [7, 3, 7],
                    'B': [4, 5, 2]})
df0['C'] = [11, 8, 9]
df0

In [ ]:
df1 = pd.DataFrame({'A': [7, 3, 7],
                    'B': [4, 5, 2]})
df1['C'] = df1['A'] + df1['B']
df1

In [ ]:
df2 = pd.DataFrame({'A': [7, 3, 7],
                    'B': [4, 5, 2]})
df2 = df2.assign(C=lambda x: x['A'] + x['B'])
df2

In [ ]:
df3 = pd.DataFrame({'A': [7, 3, 7],
                    'B': [4, 5, 2]})
df3['C'] = df3['A'].apply(lambda x: x * 2)
df3

## 7. Desafios com a base de alunos
Base com nome, idade, nota e se o(a) aluno(a) foi aprovado(a). Carregamos uma vez e cada desafio parte de uma cópia.

In [ ]:
url_alunos = 'https://raw.githubusercontent.com/alura-cursos/pandas-conhecendo-a-biblioteca/main/desafios/alunos.csv'
alunos = pd.read_csv(url_alunos, sep=',')
alunos.head()

### 7.1 Desafio 1: nulos, remoção de alunos, aprovados e substituição de notas
1. Verificar e tratar os valores nulos.
2. Remover os alunos Alice e Carlos.
3. Manter apenas os aprovados.
4. Substituir a nota 7.0 por 8.0 (correção de nota).
5. Salvar o resultado em CSV.

In [ ]:
df_alunos = alunos.copy()
df_alunos.isnull().sum()

In [ ]:
df_alunos = df_alunos.dropna()

remove_nomes = ['Carlos', 'Alice']
df_alunos = df_alunos.query('@remove_nomes not in Nome')

aprovados = df_alunos.Aprovado == 1
df_alunos = df_alunos[aprovados].copy()
df_alunos

In [ ]:
# Substitui a nota 7.0 por 8.0
# (corrigido: o original usava replace({7.0, 8.0}, inplace=True) — um set, não um dict — sobre uma coluna
#  selecionada, o que não expressa a troca 7.0 -> 8.0; o dict {antigo: novo} e a atribuição fazem a troca)
df_alunos['Notas'] = df_alunos['Notas'].replace({7.0: 8.0})
df_alunos

In [ ]:
df_alunos.to_csv('alunos_aprovados.csv', index=False, sep=',')

### 7.2 Desafio 2: pontos extras
Os alunos ganharam pontos extras equivalentes a 40% da nota. Criamos as colunas `Pontos_Extras`, `Notas_Finais` e `Aprovado_final` (nota final >= 6) e selecionamos quem não tinha sido aprovado antes, mas foi aprovado com os pontos extras.

In [ ]:
df_extras = alunos.copy()
df_extras['Pontos_Extras'] = df_extras['Notas'] * 0.4
df_extras['Notas_Finais'] = df_extras['Notas'] + df_extras['Pontos_Extras']
df_extras['Aprovado_final'] = df_extras['Notas_Finais'].apply(lambda x: True if x >= 6 else False)
df_extras.head()

In [ ]:
aprovados_com_extras = (df_extras['Aprovado_final'] == True) & (df_extras['Aprovado'] == False)
df_extras[aprovados_com_extras]